# Adult Census Income — Responsible AI Stress Tests

**Purpose:** Reproducibly compare OLS and logistic regression, investigate subgroup effects, test omitted-variable bias, inspect confident errors, and evaluate fairness metrics.

**Dataset:** UCI Adult / Census Income. This notebook combines the official `adult.data` and `adult.test` files, normalizes their target labels, and then creates one stratified 70/30 train/test split (RANDOM_STATE = 15) so every model uses the same split.

> **Important interpretation note:** This dataset is historical census data, not a validated tool for judging an individual's worth or making consequential decisions. Fairness ratios can be unstable when denominators are small or zero. Treat results as a classroom audit, not evidence that deployment is appropriate.

## Run instructions
1. Run all cells from top to bottom in Jupyter or Google Colab.
2. Internet access is needed in the first data-loading cell.
3. The notebook prints the exact values to use in your report and saves tables/plots to `adult_income_outputs/`.
4. Do not write report numbers until you have run the notebook. Every reported number should come from these outputs.


## 1. Imports and reproducibility

In [ ]:
# If needed in a fresh environment, run this in a separate cell:
# %pip install pandas numpy scikit-learn statsmodels matplotlib seaborn

import io
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)
from IPython.display import display

warnings.filterwarnings("ignore")
RANDOM_STATE = 15  # Required by the instructor
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = Path("adult_income_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Random state:", RANDOM_STATE)
print("Output folder:", OUTPUT_DIR.resolve())

## 2. Load and inspect the UCI data

The original files use `?` for missing categorical values. The test file has a one-line header and income labels ending in a period; the code below handles both. We combine the official train/test files and create a fresh stratified 70/30 split as specified by the assignment.

In [ ]:
COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country",
    "income"
]

TRAIN_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
TEST_URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

adult_train = pd.read_csv(
    TRAIN_URL, names=COLUMNS, sep=",", skipinitialspace=True,
    na_values="?", comment="|"
)
adult_test = pd.read_csv(
    TEST_URL, names=COLUMNS, sep=",", skipinitialspace=True,
    na_values="?", comment="|", skiprows=1
)

df = pd.concat([adult_train, adult_test], ignore_index=True)
df["income"] = (
    df["income"].astype(str).str.strip().str.replace(".", "", regex=False)
)
df = df[df["income"].isin([">50K", "<=50K"])].copy()
df["income_binary"] = (df["income"] == ">50K").astype(int)

print("Combined dataset shape:", df.shape)
print("\nTarget counts:")
print(df["income"].value_counts(dropna=False))
print("\nMissing values before preprocessing:")
print(df.isna().sum().sort_values(ascending=False))
display(df.head())


## 3. Preprocessing decisions

- **Missing values:** missing categorical entries are replaced with the explicit category `Unknown`, retaining rows rather than silently deleting them.
- **Categorical encoding:** one-hot encoding with a reference category (`drop_first=True`) is used for regression models.
- **Scaling:** numeric predictors are standardized using means and standard deviations learned from the training set only. This helps numerical stability; it does not change the target.
- **Outliers:** inspect numeric ranges and boxplots. Do not automatically delete valid high incomes, ages, hours, or capital gains merely because they are extreme. The Adult data's `fnlwgt` is a survey weight; for this assignment it is treated as a feature, and this choice should be disclosed.
- **Leakage control:** split before fitting imputers/scalers/encoders. The held-out test set is not used to fit preprocessing.


In [ ]:
TARGET = "income_binary"
SENSITIVE = "sex"  # Primary fairness comparison: Male vs Female
NUMERIC = [
    "age", "fnlwgt", "education_num", "capital_gain",
    "capital_loss", "hours_per_week"
]
CATEGORICAL = [
    "workclass", "education", "marital_status", "occupation",
    "relationship", "race", "sex", "native_country"
]
FEATURES = NUMERIC + CATEGORICAL

# Preserve missing values for train-only imputation; standardize numeric features.
X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", __import__("sklearn").preprocessing.OneHotEncoder(
        handle_unknown="ignore", drop="first", sparse_output=False
    ))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC),
    ("cat", categorical_pipe, CATEGORICAL)
], verbose_feature_names_out=False)

X_train_arr = preprocessor.fit_transform(X_train)
X_test_arr = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

X_train_enc = pd.DataFrame(X_train_arr, columns=feature_names, index=X_train.index)
X_test_enc = pd.DataFrame(X_test_arr, columns=feature_names, index=X_test.index)

print(f"Train rows: {len(X_train):,} ({len(X_train)/len(df):.1%})")
print(f"Test rows:  {len(X_test):,} ({len(X_test)/len(df):.1%})")
print("Encoded feature count:", len(feature_names))
print("Missing values after preprocessing (encoded train):", int(X_train_enc.isna().sum().sum()))

# Numeric summary and optional boxplots for report documentation
display(df[NUMERIC].describe().T)
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for col, ax in zip(NUMERIC, axes.flat):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "numeric_boxplots.png", dpi=160, bbox_inches="tight")
plt.show()


## Part 1 — OLS vs logistic regression

Model A uses ordinary least squares on the binary target and classifies a test row as high-risk when its predicted value is at least the **median OLS prediction on the test set**. Model B uses logistic regression and a fixed probability threshold of **0.50**. The confusion matrix below compares the two *classification rules with each other*, not either model against the true income label.

In [ ]:
ols = LinearRegression()
ols.fit(X_train_enc, y_train)
ols_pred = ols.predict(X_test_enc)
ols_cutoff = float(np.median(ols_pred))
ols_class = (ols_pred >= ols_cutoff).astype(int)

logit = LogisticRegression(max_iter=3000, solver="lbfgs", random_state=RANDOM_STATE)
logit.fit(X_train_enc, y_train)
logit_prob = logit.predict_proba(X_test_enc)[:, 1]
logit_class = (logit_prob >= 0.50).astype(int)

# Rows = OLS decision (0, 1); columns = logistic decision (0, 1)
agreement_cm = confusion_matrix(ols_class, logit_class, labels=[0, 1])
mismatch = ols_class != logit_class
mismatch_n = int(mismatch.sum())
mismatch_pct = 100 * mismatch.mean()

print(f"Median OLS predicted value (test set): {ols_cutoff:.6f}")
print(f"Logistic classification threshold: 0.500000")
print(f"Mismatched individuals: {mismatch_n:,} / {len(y_test):,} = {mismatch_pct:.2f}%")
print("Confusion matrix: rows=OLS class [low, high], columns=logistic class [low, high]")
display(pd.DataFrame(
    agreement_cm,
    index=["OLS low risk (0)", "OLS high risk (1)"],
    columns=["Logistic low risk (0)", "Logistic high risk (1)"]
))
print("\nSeparate predictive performance against the actual target (for context):")
print("OLS threshold accuracy:", round(accuracy_score(y_test, ols_class), 4))
print("Logistic accuracy:", round(accuracy_score(y_test, logit_class), 4))
print("Logistic ROC-AUC:", round(roc_auc_score(y_test, logit_prob), 4))

part1 = pd.DataFrame({
    "actual_income_binary": y_test.to_numpy(),
    "ols_prediction": ols_pred,
    "ols_class_median_threshold": ols_class,
    "logistic_probability": logit_prob,
    "logistic_class_0.5_threshold": logit_class,
    "mismatch": mismatch
}, index=y_test.index)
part1.to_csv(OUTPUT_DIR / "part1_test_predictions.csv", index=True)


## Helper: statsmodels logistic regression with confidence intervals

For interpretable coefficients and confidence intervals, the following helper fits a binomial GLM on standardized numeric predictors and one-hot encoded categorical predictors. Coefficients are log-odds changes; exponentiating a coefficient gives an odds ratio. These coefficients are for explanatory analysis, distinct from scikit-learn's predictive model above.

In [ ]:
# Stable explanatory GLM used for Parts 2.1 and 2.2.
# IMPORTANT: education and education_num encode essentially the same information,
# so education category dummies are intentionally excluded from this explanatory model.
# The predictive sklearn model in Part 1 still uses the full feature set.

EXPL_NUMERIC = ["age", "education_num", "hours_per_week", "capital_gain", "capital_loss"]
EXPL_CATEGORICAL = ["sex"]
EXPL_FEATURES = EXPL_NUMERIC + EXPL_CATEGORICAL

train_raw = X_train[EXPL_FEATURES].copy()
train_y = y_train.copy()
test_raw = X_test[EXPL_FEATURES].copy()
test_y = y_test.copy()

# Use one fixed scaling reference (the entire training set) so coefficients remain
# comparable when predictors are omitted or the model is fit on a subgroup.
EXPL_MEANS = train_raw[EXPL_NUMERIC].apply(pd.to_numeric, errors="coerce").mean()
EXPL_STDS = train_raw[EXPL_NUMERIC].apply(pd.to_numeric, errors="coerce").std(ddof=0).replace(0, 1).fillna(1)

def make_design(data, exclude=None, include_sex=True):
    """Build a stable design matrix with train-set scaling and reference coding for sex."""
    exclude = set(exclude or [])
    parts = []
    numeric_cols = [c for c in EXPL_NUMERIC if c not in exclude]
    if numeric_cols:
        nums = data[numeric_cols].copy()
        for col in numeric_cols:
            nums[col] = pd.to_numeric(nums[col], errors="coerce")
            nums[col] = nums[col].fillna(EXPL_MEANS[col])
            nums[col] = (nums[col] - EXPL_MEANS[col]) / EXPL_STDS[col]
        parts.append(nums.astype(float))
    if include_sex and "sex" not in exclude and "sex" in data.columns:
        sex = data["sex"].fillna("Unknown").astype(str)
        # Male = 1, Female = 0; unknown values are treated as 0 and should be noted if present.
        parts.append(pd.DataFrame({"sex_Male": (sex == "Male").astype(float)}, index=data.index))
    design = pd.concat(parts, axis=1) if parts else pd.DataFrame(index=data.index)
    design = sm.add_constant(design, has_constant="add")
    return design.astype(float)

def fit_glm(data, target, exclude=None, include_sex=True):
    design = make_design(data, exclude=exclude, include_sex=include_sex)
    target_aligned = target.loc[design.index].astype(float)
    result = sm.GLM(target_aligned, design, family=sm.families.Binomial()).fit(maxiter=200)
    if not getattr(result, "converged", True):
        raise RuntimeError("Explanatory logistic regression did not converge. Check model inputs.")
    if not np.isfinite(result.params.to_numpy()).all():
        raise RuntimeError("Non-finite GLM coefficients detected; inspect the design matrix.")
    return result, design

full_glm, full_design = fit_glm(train_raw, train_y)
print("Stable explanatory GLM converged:", full_glm.converged)
print("Training rows:", len(train_raw))
print("Predictors:", list(full_design.columns))
print("Log-likelihood:", round(float(full_glm.llf), 3))

## Part 2.1 — Subgroup effect / Simpson's paradox hunt

We compare the coefficient for standardized `education_num` in the full training model and in separate Male/Female models. If the coefficient changes sign, that is evidence of a reversal for this specification. If it does not reverse, report the actual difference in magnitude and confidence intervals rather than claiming Simpson's paradox. Separate subgroup coefficients are associations, not causal effects.

In [ ]:
# Part 2.1: estimate education_num association for the full training set and each sex subgroup.
subgroup_results = {}
for group in ["Male", "Female"]:
    idx = train_raw["sex"].astype(str) == group
    if int(idx.sum()) == 0:
        print(f"Skipping empty subgroup: {group}")
        continue
    result, _ = fit_glm(train_raw.loc[idx], train_y.loc[idx], exclude=["sex"], include_sex=False)
    subgroup_results[group] = result
    print(f"{group} model converged: {result.converged}; rows={int(idx.sum()):,}")

def coefficient_row(result, term):
    if term not in result.params.index:
        return {"term": term, "coef": np.nan, "ci_low": np.nan, "ci_high": np.nan, "p_value": np.nan}
    ci = result.conf_int().loc[term]
    return {
        "term": term,
        "coef": float(result.params[term]),
        "ci_low": float(ci.iloc[0]),
        "ci_high": float(ci.iloc[1]),
        "p_value": float(result.pvalues[term])
    }

rows = [{"model": "Full training set", **coefficient_row(full_glm, "education_num")}]
for group, result in subgroup_results.items():
    rows.append({"model": f"Subgroup: {group}", **coefficient_row(result, "education_num")})
subgroup_table = pd.DataFrame(rows)
display(subgroup_table.round(6))
subgroup_table.to_csv(OUTPUT_DIR / "subgroup_coefficients.csv", index=False)

coef_values = subgroup_table["coef"].dropna()
if len(coef_values) >= 2 and (coef_values > 0).any() and (coef_values < 0).any():
    print("Sign reversal detected for education_num across the reported models. Investigate before claiming Simpson's paradox.")
else:
    print("No sign reversal in these models. Compare magnitudes and confidence intervals; subgroup coefficient differences alone do not prove Simpson's paradox.")
print("Interpretation: education_num is standardized using the full training-set scale, so its coefficient is the log-odds change for a one-standard-deviation increase, controlling for the other included predictors.")

## Part 2.2 — Omitted-variable bias

We deliberately omit `marital_status`, a plausible confounder because marital status can be associated with both employment/income patterns and other predictors. Compare four pre-specified coefficients before and after omission. Since the design is standardized separately within each fitted sample/design, compare the listed coefficients as model-specific standardized associations and disclose this convention. The percentage change uses absolute full-model coefficient as denominator; it is undefined if the full coefficient is zero.

In [ ]:
# Part 2.2: omit one predictor at a time and compare the coefficients of predictors retained.
# All numeric features use the same training-set scaling in every model.

bias_rows = []
for omitted in EXPL_NUMERIC + ["sex"]:
    reduced_exclude = [omitted]
    reduced_glm, reduced_design = fit_glm(
        train_raw, train_y, exclude=reduced_exclude,
        include_sex=(omitted != "sex")
    )
    common_terms = [term for term in full_glm.params.index if term in reduced_glm.params.index and term != "const"]
    for term in common_terms:
        full_b = float(full_glm.params[term])
        reduced_b = float(reduced_glm.params[term])
        pct = ((reduced_b - full_b) / abs(full_b) * 100) if abs(full_b) > 1e-12 else np.nan
        bias_rows.append({
            "omitted_predictor": omitted,
            "compared_predictor": term,
            "full_model_coef": full_b,
            "reduced_model_coef": reduced_b,
            "absolute_change": abs(reduced_b - full_b),
            "percentage_change": pct,
            "full_p_value": float(full_glm.pvalues.get(term, np.nan)),
            "reduced_p_value": float(reduced_glm.pvalues.get(term, np.nan))
        })

bias_table = pd.DataFrame(bias_rows)
display(bias_table.round(5))
bias_table.to_csv(OUTPUT_DIR / "omitted_variable_bias.csv", index=False)

if not bias_table.empty:
    largest = bias_table.loc[bias_table["absolute_change"].idxmax()]
    print(
        f"Largest absolute coefficient shift: omitting {largest['omitted_predictor']} changed "
        f"{largest['compared_predictor']} from {largest['full_model_coef']:.5f} "
        f"to {largest['reduced_model_coef']:.5f} "
        f"(absolute change {largest['absolute_change']:.5f}; "
        f"percentage change {largest['percentage_change']:.2f}%)."
    )
print("Caution: coefficient changes in logistic regression can also reflect odds-ratio non-collapsibility; they do not by themselves prove causal omitted-variable bias.")

## Part 2.3 — Adversarial subset: confident wrong predictions

Use the scikit-learn logistic model's held-out probabilities. A confident error is either a false positive with probability at least 0.75, or a false negative with probability at most 0.25. If there are more than 50, a fixed random seed selects 50; if fewer, use all. The characteristic scan compares the selected rows with the full test set. Because the selection itself depends on errors, patterns are descriptive and should not be presented as causal explanations.

In [ ]:
adversarial_mask = ((logit_prob >= 0.75) & (y_test.to_numpy() == 0)) | ((logit_prob <= 0.25) & (y_test.to_numpy() == 1))
adv_positions = np.flatnonzero(adversarial_mask)
rng = np.random.default_rng(RANDOM_STATE)
if len(adv_positions) > 50:
    adv_positions = np.sort(rng.choice(adv_positions, size=50, replace=False))

adv_idx = y_test.index[adv_positions]
adv = X_test.loc[adv_idx].copy()
adv["actual_income_binary"] = y_test.loc[adv_idx]
adv["predicted_probability"] = logit_prob[adv_positions]
adv["predicted_class"] = logit_class[adv_positions]
adv["error_type"] = np.where(
    (adv["predicted_probability"] >= 0.75) & (adv["actual_income_binary"] == 0),
    "False positive", "False negative"
)
adv["confidence"] = np.where(adv["error_type"] == "False positive",
                             adv["predicted_probability"], 1 - adv["predicted_probability"])

print(f"All confident wrong predictions: {int(adversarial_mask.sum()):,} / {len(y_test):,} = {100*adversarial_mask.mean():.2f}% of test set")
print(f"Rows selected for subset analysis: {len(adv):,}")
if len(adv):
    print("\nError type breakdown:")
    display(adv["error_type"].value_counts().rename_axis("error_type").to_frame("count"))
    print("\nSubset demographics:")
    for col in ["age", "sex", "race", "education", "marital_status", "occupation"]:
        if col == "age":
            print("age — subset median/mean:", round(adv[col].median(), 2), "/", round(adv[col].mean(), 2),
                  "| test median/mean:", round(X_test[col].median(), 2), "/", round(X_test[col].mean(), 2))
        else:
            print(f"\n{col} — subset (%):")
            display((adv[col].fillna("Unknown").value_counts(normalize=True).head(8)*100).round(1).to_frame("subset_pct"))
            print(f"{col} — full test (%):")
            display((X_test[col].fillna("Unknown").value_counts(normalize=True).head(8)*100).round(1).to_frame("test_pct"))
    # Automatically rank common categorical traits within the subset.
    trait_rows = []
    for col in ["sex", "race", "education", "marital_status", "occupation", "workclass", "relationship"]:
        if len(adv):
            vals = adv[col].fillna("Unknown").astype(str).value_counts(normalize=True)
            top_value = vals.index[0]
            test_share = (X_test[col].fillna("Unknown").astype(str) == top_value).mean()
            trait_rows.append({"feature": col, "most_common_subset_value": top_value,
                               "subset_share_pct": 100*vals.iloc[0],
                               "test_set_share_of_same_value_pct": 100*test_share})
    trait_table = pd.DataFrame(trait_rows).sort_values("subset_share_pct", ascending=False)
    print("\nMost common subset traits (compare with test-set prevalence before drawing conclusions):")
    display(trait_table.round(2))
    adv.to_csv(OUTPUT_DIR / "adversarial_subset.csv", index=True)
    trait_table.to_csv(OUTPUT_DIR / "adversarial_trait_summary.csv", index=False)
else:
    print("No predictions met the specified confidence/error criteria. Report this result; do not fabricate a 50-row subset.")


## Part 2.4 — Fairness threshold sweep

Sensitive attribute: `sex`, comparing Male (Group A) and Female (Group B). Metrics are calculated on the held-out test set for thresholds 0.05 to 0.95 by 0.10.

- **DP ratio:** selection rate A / selection rate B.
- **EO summary ratio:** `min(TPR_A / TPR_B, FPR_A / FPR_B)` as specified in the assignment. We also retain the underlying rates because a single ratio can conceal which error rate is driving the result.
- **PP ratio:** precision A / precision B among predicted positives.

A ratio of 1 is parity. Ratios may be undefined when a denominator is zero, and values above 1.5 are valid even though the requested chart axes stop at 1.5.

In [ ]:
def safe_ratio(a, b):
    if pd.isna(a) or pd.isna(b) or abs(b) < 1e-12:
        return np.nan
    return a / b

def group_rates(y_true, y_pred, group_mask):
    yt = np.asarray(y_true)[group_mask]
    yp = np.asarray(y_pred)[group_mask]
    positives = yp == 1
    actual_pos = yt == 1
    actual_neg = yt == 0
    selection = float(np.mean(yp == 1)) if len(yp) else np.nan
    tpr = float(np.sum(positives & actual_pos) / np.sum(actual_pos)) if np.sum(actual_pos) else np.nan
    fpr = float(np.sum(positives & actual_neg) / np.sum(actual_neg)) if np.sum(actual_neg) else np.nan
    precision = float(np.sum(positives & actual_pos) / np.sum(positives)) if np.sum(positives) else np.nan
    return {"selection_rate": selection, "TPR": tpr, "FPR": fpr, "precision": precision}

test_sex = X_test["sex"].astype(str).to_numpy()
y_test_arr = y_test.to_numpy()
thresholds = np.round(np.arange(0.05, 0.951, 0.10), 2)
fair_rows = []
for threshold in thresholds:
    pred = (logit_prob >= threshold).astype(int)
    A = group_rates(y_test_arr, pred, test_sex == "Male")
    B = group_rates(y_test_arr, pred, test_sex == "Female")
    dp = safe_ratio(A["selection_rate"], B["selection_rate"])
    tpr_ratio = safe_ratio(A["TPR"], B["TPR"])
    fpr_ratio = safe_ratio(A["FPR"], B["FPR"])
    eo = min(tpr_ratio, fpr_ratio) if np.isfinite(tpr_ratio) and np.isfinite(fpr_ratio) else np.nan
    pp = safe_ratio(A["precision"], B["precision"])
    fair_rows.append({
        "threshold": threshold, "DP_ratio": dp, "EO_ratio": eo, "PP_ratio": pp,
        "selection_A_male": A["selection_rate"], "selection_B_female": B["selection_rate"],
        "TPR_A_male": A["TPR"], "TPR_B_female": B["TPR"],
        "FPR_A_male": A["FPR"], "FPR_B_female": B["FPR"],
        "precision_A_male": A["precision"], "precision_B_female": B["precision"],
        "accuracy": accuracy_score(y_test_arr, pred)
    })
fairness = pd.DataFrame(fair_rows)
display(fairness.round(4))
fairness.to_csv(OUTPUT_DIR / "fairness_threshold_sweep.csv", index=False)

plt.figure(figsize=(9, 7))
plt.scatter(fairness["DP_ratio"], fairness["EO_ratio"])
for _, r in fairness.iterrows():
    if np.isfinite(r["DP_ratio"]) and np.isfinite(r["EO_ratio"]):
        plt.annotate(f'{r["threshold"]:.2f}', (r["DP_ratio"], r["EO_ratio"]),
                     xytext=(4, 4), textcoords="offset points", fontsize=8)
plt.xlim(0, 1.5)
plt.ylim(0, 1.5)
plt.xlabel("Demographic Parity ratio (Male / Female)")
plt.ylabel("Equalized Odds summary ratio")
plt.title("Fairness trade-off across logistic decision thresholds")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fairness_tradeoff_chart.png", dpi=180, bbox_inches="tight")
plt.show()

# Print best thresholds by closeness to parity, plus a concrete pairwise DP-vs-EO tradeoff.
fairness["DP_distance_from_1"] = (fairness["DP_ratio"] - 1).abs()
fairness["EO_distance_from_1"] = (fairness["EO_ratio"] - 1).abs()
print("Threshold closest to DP parity:")
display(fairness.loc[[fairness["DP_distance_from_1"].idxmin()],
                     ["threshold", "DP_ratio", "EO_ratio", "PP_ratio", "accuracy"]].round(4))
print("Threshold closest to EO-summary parity:")
display(fairness.loc[[fairness["EO_distance_from_1"].idxmin()],
                     ["threshold", "DP_ratio", "EO_ratio", "PP_ratio", "accuracy"]].round(4))

valid = fairness.dropna(subset=["DP_ratio", "EO_ratio"]).copy()
if len(valid) >= 2:
    # Choose the pair with greatest DP improvement and report accompanying EO movement.
    pairs = []
    for i, r1 in valid.iterrows():
        for j, r2 in valid.iterrows():
            if r2["DP_ratio"] < r1["DP_ratio"] and r2["DP_ratio"] <= 1:
                dp_gain = abs(r1["DP_ratio"] - 1) - abs(r2["DP_ratio"] - 1)
                eo_loss = abs(r2["EO_ratio"] - 1) - abs(r1["EO_ratio"] - 1)
                pairs.append((dp_gain, eo_loss, r1, r2))
    if pairs:
        best = max(pairs, key=lambda x: x[0])
        dp_gain, eo_loss, r1, r2 = best
        print(f"Illustrative threshold shift {r1['threshold']:.2f} -> {r2['threshold']:.2f}: "
              f"DP ratio {r1['DP_ratio']:.3f} -> {r2['DP_ratio']:.3f}; "
              f"EO ratio {r1['EO_ratio']:.3f} -> {r2['EO_ratio']:.3f}.")
        print("Use the actual table to explain whether the EO distance from 1 increased or decreased; do not describe a change as a gain unless it moves closer to 1.")


## Part 3 — Report-ready result bundle

This cell exports compact results that can be cited in the written answers. Use the generated values and explain limitations in plain language. Avoid claiming a causal relationship from observational data, and avoid calling an effect a Simpson's paradox unless the sign reversal is actually present.

In [ ]:
# Export core model summary and coefficient tables.
model_summary = pd.DataFrame([
    {"model": "OLS + median threshold", "threshold": ols_cutoff,
     "accuracy_vs_true": accuracy_score(y_test, ols_class),
     "precision_vs_true": precision_score(y_test, ols_class, zero_division=0),
     "recall_vs_true": recall_score(y_test, ols_class, zero_division=0),
     "f1_vs_true": f1_score(y_test, ols_class, zero_division=0)},
    {"model": "Logistic + 0.50 threshold", "threshold": 0.50,
     "accuracy_vs_true": accuracy_score(y_test, logit_class),
     "precision_vs_true": precision_score(y_test, logit_class, zero_division=0),
     "recall_vs_true": recall_score(y_test, logit_class, zero_division=0),
     "f1_vs_true": f1_score(y_test, logit_class, zero_division=0)}
])
model_summary.to_csv(OUTPUT_DIR / "model_summary.csv", index=False)
display(model_summary.round(4))

# A compact coefficient extract for explaining model behavior.
coef_ci = full_glm.conf_int()
coef_extract = pd.DataFrame({
    "coefficient": full_glm.params,
    "ci_low": coef_ci[0],
    "ci_high": coef_ci[1],
    "p_value": full_glm.pvalues
}).sort_values("p_value")
coef_extract.to_csv(OUTPUT_DIR / "full_model_coefficients.csv", index=True)
print("Most statistically significant terms in explanatory GLM (association, not causation):")
display(coef_extract.head(15).round(5))

print("\nSaved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print("-", path.name)


## Interpretation checklist for the PDF

Before submission, verify that your report contains:

1. **Preprocessing:** combined data source, missing-value strategy, categorical encoding, scaling, outlier inspection, and 70/30 split.
2. **Part 1:** median OLS prediction, logistic threshold 0.50, model-agreement confusion matrix, mismatch count and percentage.
3. **Part 2.1:** full and subgroup education coefficients, 95% confidence intervals, p-values, and an honest statement about whether signs reverse.
4. **Part 2.2:** omitted variable, at least four before/after coefficients, percentage shifts, and largest shift.
5. **Part 2.3:** total confident-error count and percentage, selected subset size, demographics compared with the test set, and the most common traits.
6. **Part 2.4:** all 10 threshold rows, DP/EO/PP ratios, trade-off plot, and a defensible threshold choice with counterargument.
7. **Parts 3–4:** every answer cites actual outputs from this run. Do not insert example numbers as if they were results.

### Technical caveats worth acknowledging
- The specified fairness ratios can be undefined if a group's denominator is zero; report `NaN`/undefined rather than fabricating a ratio.
- `min(TPR_A/TPR_B, FPR_A/FPR_B)` is the assignment's requested EO summary, but it is not the only convention used for equalized odds. Include the group TPR and FPR columns so readers can see both dimensions.
- The median-OLS threshold and the logistic 0.50 threshold intentionally use different rules, so disagreement is expected and is not by itself proof that either model is better.
- Statistical significance is not the same as practical importance, and coefficient differences do not prove causality.
